<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/02_data_preprocessing_and_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — Data Preprocessing and Feature Engineering

## Financial Fraud Detection: IEEE-CIS and PaySim

This notebook converts the raw datasets into reproducible, leakage-aware train/validation/test datasets for the downstream feature-selection and modelling notebooks.

### Key design decisions
- IEEE-CIS and PaySim are processed independently.
- IEEE-CIS uses a chronological 70% / 15% / 15% split based on `TransactionDT`.
- PaySim uses a chronological 70% / 15% / 15% split based on `step`.
- Raw categorical columns are retained in the IEEE-CIS split files because next notebook learns frequency encoding, imputation, scaling, and one-hot encoding from the training split.
- All preprocessing statistics and category mappings must be learned from training data only.
- The target and identifiers are stored separately from feature matrices.
- No oversampling is performed here. Resampling, if tested, belongs inside model training and must never touch validation or test data.

## 1. Imports and reproducibility

In [1]:
import os
import gc
import json
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

RANDOM_STATE = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

assert abs(TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION - 1.0) < 1e-9

print("Libraries imported.")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Libraries imported.
Pandas: 2.2.3
NumPy: 2.4.6


## 2. Mount Drive and define paths

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
BASE_DIR = Path("/content/drive/MyDrive/fraud-detection-research")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROCESSED_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PAYSIM_FILE = DATA_DIR / "PS_20174392719_1491204439457_log.csv"
IEEE_TRANSACTION_FILE = DATA_DIR / "train_transaction.csv"
IEEE_IDENTITY_FILE = DATA_DIR / "train_identity.csv"

print("Data directory:", DATA_DIR)
print("Processed directory:", PROCESSED_DIR)

for file_path in [PAYSIM_FILE, IEEE_TRANSACTION_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}\n"
            "Run Notebook 01 first; it extracts/copies the raw CSV files here."
        )

if not IEEE_IDENTITY_FILE.exists():
    print("WARNING: train_identity.csv not found. IEEE-CIS will be processed without identity features.")

Data directory: /content/drive/MyDrive/fraud-detection-research/data
Processed directory: /content/drive/MyDrive/fraud-detection-research/data/processed


## 3. Helper functions

In [4]:
def report_class_distribution(y, name):
    y = pd.Series(y).reset_index(drop=True)
    counts = y.value_counts().reindex([0, 1], fill_value=0)
    report = pd.DataFrame({
        "Count": counts,
        "Percentage": counts / len(y) * 100
    }, index=["Legitimate (0)", "Fraud (1)"])
    print(f"\n{name}: {len(y):,} rows")
    display(report.round(4))
    if counts.loc[1] > 0:
        print(f"Legitimate : fraud ratio = {counts.loc[0] / counts.loc[1]:,.1f}:1")
    return report


def chronological_split_indices(time_values, train_fraction=0.70, val_fraction=0.15):
    """Create non-overlapping chronological splits using row positions.

    Sorting is stable. The split is row-count based, avoiding quantile-boundary
    overlap when many transactions share the same timestamp/step.
    """
    time_values = pd.Series(time_values).reset_index(drop=True)
    if time_values.isna().any():
        raise ValueError("Time column contains missing values; resolve before splitting.")

    order = np.argsort(time_values.to_numpy(), kind="mergesort")
    n = len(order)
    train_end = int(np.floor(n * train_fraction))
    val_end = int(np.floor(n * (train_fraction + val_fraction)))

    train_idx = order[:train_end]
    val_idx = order[train_end:val_end]
    test_idx = order[val_end:]

    assert len(train_idx) + len(val_idx) + len(test_idx) == n
    assert len(set(train_idx).intersection(val_idx)) == 0
    assert len(set(train_idx).intersection(test_idx)) == 0
    assert len(set(val_idx).intersection(test_idx)) == 0

    return train_idx, val_idx, test_idx


def save_split_pair(X, y, prefix, split_name, output_dir=PROCESSED_DIR):
    X_path = output_dir / f"{prefix}_X_{split_name}.parquet"
    y_path = output_dir / f"{prefix}_y_{split_name}.parquet"
    X.to_parquet(X_path, index=False)
    pd.Series(y, name="isFraud").to_frame().to_parquet(y_path, index=False)
    print(f"Saved {X_path.name}: {X.shape}; {y_path.name}: {len(y):,} labels")


def verify_split_files(prefix, split_names=("train", "validation", "test")):
    for split in split_names:
        x_path = PROCESSED_DIR / f"{prefix}_X_{split}.parquet"
        y_path = PROCESSED_DIR / f"{prefix}_y_{split}.parquet"
        if not x_path.exists() or not y_path.exists():
            raise FileNotFoundError(f"Missing split files for {prefix} / {split}")
        X = pd.read_parquet(x_path)
        y = pd.read_parquet(y_path)
        assert len(X) == len(y), f"Row mismatch for {prefix} {split}"
        print(f"{prefix} {split}: X={X.shape}, y={y.shape}, fraud_rate={y.iloc[:, 0].mean()*100:.4f}%")

## 4. Load and merge IEEE-CIS

In [5]:
ieee_transaction = pd.read_csv(IEEE_TRANSACTION_FILE, low_memory=False)
print("Transaction table:", ieee_transaction.shape)

if IEEE_IDENTITY_FILE.exists():
    ieee_identity = pd.read_csv(IEEE_IDENTITY_FILE, low_memory=False)
    print("Identity table:", ieee_identity.shape)

    if ieee_transaction["TransactionID"].duplicated().any():
        raise ValueError("Duplicate TransactionID values found in transaction table.")
    if ieee_identity["TransactionID"].duplicated().any():
        raise ValueError("Duplicate TransactionID values found in identity table.")

    ieee_data = ieee_transaction.merge(
        ieee_identity,
        on="TransactionID",
        how="left",
        validate="one_to_one"
    )
    del ieee_identity
else:
    ieee_data = ieee_transaction.copy()

del ieee_transaction
gc.collect()

required = {"TransactionID", "TransactionDT", "isFraud"}
missing = required - set(ieee_data.columns)
if missing:
    raise ValueError(f"IEEE-CIS missing required columns: {missing}")

print("Merged IEEE-CIS shape:", ieee_data.shape)
print("Target values:", sorted(ieee_data["isFraud"].dropna().unique().tolist()))

Transaction table: (590540, 394)
Identity table: (144233, 41)
Merged IEEE-CIS shape: (590540, 434)
Target values: [0, 1]


## 5. IEEE-CIS data-quality checks

In [6]:
# Keep one row per transaction. Do not deduplicate on feature values because
# different transactions may legitimately share identical feature values.
duplicate_ids = int(ieee_data["TransactionID"].duplicated().sum())
print("Duplicate TransactionID rows:", duplicate_ids)
if duplicate_ids:
    ieee_data = ieee_data.drop_duplicates(subset="TransactionID", keep="first").copy()

# Rows without a target cannot be used for supervised training.
missing_target = int(ieee_data["isFraud"].isna().sum())
print("Rows with missing target:", missing_target)
if missing_target:
    ieee_data = ieee_data.dropna(subset=["isFraud"]).copy()

ieee_data["isFraud"] = ieee_data["isFraud"].astype("int8")
if not set(ieee_data["isFraud"].unique()).issubset({0, 1}):
    raise ValueError("IEEE-CIS target must be binary 0/1.")

print("Shape after checks:", ieee_data.shape)
print("Exact duplicate rows:", int(ieee_data.duplicated().sum()))
display(ieee_data[["TransactionID", "TransactionDT", "isFraud"]].head())
report_class_distribution(ieee_data["isFraud"], "IEEE-CIS after data-quality checks")

Duplicate TransactionID rows: 0
Rows with missing target: 0
Shape after checks: (590540, 434)
Exact duplicate rows: 0


,TransactionID,TransactionDT,isFraud
0,2987000,86400,0
1,2987001,86401,0
2,2987002,86469,0
3,2987003,86499,0
4,2987004,86506,0



IEEE-CIS after data-quality checks: 590,540 rows


,Count,Percentage
Legitimate (0),NaN,NaN
Fraud (1),NaN,NaN


Legitimate : fraud ratio = 27.6:1


,Count,Percentage
Legitimate (0),NaN,NaN
Fraud (1),NaN,NaN


## 6. IEEE-CIS chronological split

The split is based on sorted transaction time rather than random sampling. This better represents the deployment setting in which a model trained on earlier transactions is evaluated on later transactions.

**Important:** rows are split by chronological position rather than using quantile thresholds. This avoids accidental overlap when many records have the same `TransactionDT`.

In [7]:
ieee_data = ieee_data.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)

ieee_train_idx, ieee_val_idx, ieee_test_idx = chronological_split_indices(
    ieee_data["TransactionDT"],
    TRAIN_FRACTION,
    VALIDATION_FRACTION
)

ieee_train = ieee_data.iloc[ieee_train_idx].copy()
ieee_val = ieee_data.iloc[ieee_val_idx].copy()
ieee_test = ieee_data.iloc[ieee_test_idx].copy()

for name, frame in [
    ("Train", ieee_train), ("Validation", ieee_val), ("Test", ieee_test)
]:
    print(
        f"{name}: {len(frame):,} rows | "
        f"time range {frame['TransactionDT'].min():,.0f}–{frame['TransactionDT'].max():,.0f} | "
        f"fraud rate {frame['isFraud'].mean()*100:.4f}%"
    )

assert ieee_train["TransactionDT"].max() <= ieee_val["TransactionDT"].min()
assert ieee_val["TransactionDT"].max() <= ieee_test["TransactionDT"].min()

Train: 413,378 rows | time range 86,400–10,437,996 | fraud rate 3.5169%
Validation: 88,581 rows | time range 10,438,003–13,151,840 | fraud rate 3.4341%
Test: 88,581 rows | time range 13,151,880–15,811,131 | fraud rate 3.4804%


In [8]:
# Separate identifiers and target from model inputs.
# TransactionID is retained separately for traceability, not as a predictor.
ieee_target_col = "isFraud"
ieee_id_col = "TransactionID"

def split_features_labels(frame, target_col, id_col):
    y = frame[target_col].astype("int8").reset_index(drop=True)
    ids = frame[id_col].reset_index(drop=True)
    X = frame.drop(columns=[target_col, id_col]).reset_index(drop=True)
    return X, y, ids

ieee_X_train, ieee_y_train, ieee_id_train = split_features_labels(
    ieee_train, ieee_target_col, ieee_id_col
)
ieee_X_validation, ieee_y_validation, ieee_id_validation = split_features_labels(
    ieee_val, ieee_target_col, ieee_id_col
)
ieee_X_test, ieee_y_test, ieee_id_test = split_features_labels(
    ieee_test, ieee_target_col, ieee_id_col
)

print("Feature shapes:")
print("Train:", ieee_X_train.shape, ieee_y_train.shape)
print("Validation:", ieee_X_validation.shape, ieee_y_validation.shape)
print("Test:", ieee_X_test.shape, ieee_y_test.shape)

assert list(ieee_X_train.columns) == list(ieee_X_validation.columns) == list(ieee_X_test.columns)

Feature shapes:
Train: (413378, 432) (413378,)
Validation: (88581, 432) (88581,)
Test: (88581, 432) (88581,)


## 7. Save IEEE-CIS splits for Notebook 03

In [9]:
for split_name, X_split, y_split, id_split in [
    ("train", ieee_X_train, ieee_y_train, ieee_id_train),
    ("validation", ieee_X_validation, ieee_y_validation, ieee_id_validation),
    ("test", ieee_X_test, ieee_y_test, ieee_id_test)
]:
    save_split_pair(X_split, y_split, "ieee", split_name)
    id_split.to_frame("TransactionID").to_parquet(
        PROCESSED_DIR / f"ieee_id_{split_name}.parquet",
        index=False
    )

# Save a compact split manifest documenting the time boundaries.
ieee_manifest = {
    "dataset": "IEEE-CIS",
    "split_strategy": "chronological row-position split by TransactionDT",
    "fractions": {
        "train": TRAIN_FRACTION,
        "validation": VALIDATION_FRACTION,
        "test": TEST_FRACTION
    },
    "rows": {
        "train": len(ieee_train),
        "validation": len(ieee_val),
        "test": len(ieee_test)
    },
    "time_ranges": {
        "train": [float(ieee_train["TransactionDT"].min()), float(ieee_train["TransactionDT"].max())],
        "validation": [float(ieee_val["TransactionDT"].min()), float(ieee_val["TransactionDT"].max())],
        "test": [float(ieee_test["TransactionDT"].min()), float(ieee_test["TransactionDT"].max())]
    },
    "feature_columns": list(ieee_X_train.columns),
    "target": "isFraud",
    "identifier_excluded_from_features": "TransactionID"
}
with open(REPORT_DIR / "ieee_split_manifest.json", "w") as f:
    json.dump(ieee_manifest, f, indent=2)

verify_split_files("ieee")

Saved ieee_X_train.parquet: (413378, 432); ieee_y_train.parquet: 413,378 labels
Saved ieee_X_validation.parquet: (88581, 432); ieee_y_validation.parquet: 88,581 labels
Saved ieee_X_test.parquet: (88581, 432); ieee_y_test.parquet: 88,581 labels
ieee train: X=(413378, 432), y=(413378, 1), fraud_rate=3.5169%
ieee validation: X=(88581, 432), y=(88581, 1), fraud_rate=3.4341%
ieee test: X=(88581, 432), y=(88581, 1), fraud_rate=3.4804%


## 8. IEEE-CIS missingness and feature metadata

This section generates descriptive reports only. It does not drop columns or impute values globally.

In [10]:
ieee_missing_report = pd.DataFrame({
    "Missing_Count": ieee_X_train.isna().sum(),
    "Missing_Percentage": ieee_X_train.isna().mean() * 100,
    "Unique_Values": ieee_X_train.nunique(dropna=True),
    "Dtype": ieee_X_train.dtypes.astype(str)
}).sort_values("Missing_Percentage", ascending=False)

ieee_missing_report.to_csv(REPORT_DIR / "ieee_training_missingness.csv")
display(ieee_missing_report.head(30))

ieee_categorical_cols = ieee_X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

ieee_cardinality = pd.DataFrame({
    "Unique_Values": ieee_X_train[ieee_categorical_cols].nunique(dropna=True),
    "Missing_Count": ieee_X_train[ieee_categorical_cols].isna().sum()
}).sort_values("Unique_Values", ascending=False)

ieee_cardinality.to_csv(REPORT_DIR / "ieee_categorical_cardinality_train.csv")
display(ieee_cardinality.head(30))

print("Numeric columns:", len(ieee_X_train.select_dtypes(include=np.number).columns))
print("Categorical columns:", len(ieee_categorical_cols))

,Missing_Count,Missing_Percentage,Unique_Values,Dtype
id_24,409769,99.12695,10,float64
id_25,409463,99.05292,307,float64
id_07,409448,99.04930,81,float64
id_08,409448,99.04930,93,float64
id_26,409445,99.04857,85,float64
id_21,409443,99.04809,421,float64
id_23,409439,99.04712,3,object
id_27,409439,99.04712,2,object
id_22,409439,99.04712,23,float64
D7,386632,93.52989,549,float64


,Unique_Values,Missing_Count
DeviceInfo,1546,322016
id_33,183,355865
id_31,108,306142
id_30,71,351588
R_emaildomain,60,310337
P_emaildomain,59,64296
ProductCD,5,0
card6,4,826
card4,4,829
id_34,4,351519


Numeric columns: 401
Categorical columns: 31


## 9. Load and validate PaySim

In [11]:
paysim_data = pd.read_csv(PAYSIM_FILE, low_memory=False)

required_paysim = {
    "step", "type", "amount", "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest", "isFraud"
}
missing = required_paysim - set(paysim_data.columns)
if missing:
    raise ValueError(f"PaySim missing required columns: {missing}")

print("PaySim shape:", paysim_data.shape)
print("Duplicate rows:", int(paysim_data.duplicated().sum()))
print("Duplicate transaction identifiers (nameOrig):",
      int(paysim_data["nameOrig"].duplicated().sum()) if "nameOrig" in paysim_data else "N/A")

paysim_data = paysim_data.drop_duplicates().copy()
paysim_data = paysim_data.dropna(subset=["isFraud", "step"]).copy()
paysim_data["isFraud"] = paysim_data["isFraud"].astype("int8")

if not set(paysim_data["isFraud"].unique()).issubset({0, 1}):
    raise ValueError("PaySim target must be binary 0/1.")

report_class_distribution(paysim_data["isFraud"], "PaySim after data-quality checks")
display(paysim_data.head())

PaySim shape: (6362620, 11)
Duplicate rows: 0
Duplicate transaction identifiers (nameOrig): 9313

PaySim after data-quality checks: 6,362,620 rows


,Count,Percentage
Legitimate (0),NaN,NaN
Fraud (1),NaN,NaN


Legitimate : fraud ratio = 773.7:1


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64000",C1231006815,"170,136.00000","160,296.36000",M1979787155,0.00000,0.00000,0,0
1,1,PAYMENT,"1,864.28000",C1666544295,"21,249.00000","19,384.72000",M2044282225,0.00000,0.00000,0,0
2,1,TRANSFER,181.00000,C1305486145,181.00000,0.00000,C553264065,0.00000,0.00000,1,0
3,1,CASH_OUT,181.00000,C840083671,181.00000,0.00000,C38997010,"21,182.00000",0.00000,1,0
4,1,PAYMENT,"11,668.14000",C2048537720,"41,554.00000","29,885.86000",M1230701703,0.00000,0.00000,0,0


## 10. PaySim feature engineering

The following features are derived from fields in the supplied PaySim data.

- `hour_of_day` and `simulation_day` are derived from `step`.
- `log_amount` reduces the right-skew of transaction amount.
- Balance-change and balance-error features are included in the **extended/offline** feature set.
- Account names and the original `step` are excluded from the model input matrices. The step remains available separately for chronological splitting and audit.

The real-time set is deliberately conservative: it excludes destination and post-transaction balance fields. Whether all remaining fields are available at a real deployment scoring point must still be confirmed against the intended transaction workflow.

In [12]:
paysim_work = paysim_data.sort_values("step", kind="mergesort").reset_index(drop=True).copy()

paysim_work["hour_of_day"] = (paysim_work["step"] % 24).astype("int8")
paysim_work["simulation_day"] = (paysim_work["step"] // 24).astype("int16")
paysim_work["log_amount"] = np.log1p(paysim_work["amount"].clip(lower=0))

paysim_work["origin_balance_change"] = (
    paysim_work["oldbalanceOrg"] - paysim_work["newbalanceOrig"]
)
paysim_work["destination_balance_change"] = (
    paysim_work["newbalanceDest"] - paysim_work["oldbalanceDest"]
)
paysim_work["origin_balance_error"] = (
    paysim_work["oldbalanceOrg"] - paysim_work["amount"] - paysim_work["newbalanceOrig"]
)
paysim_work["destination_balance_error"] = (
    paysim_work["oldbalanceDest"] + paysim_work["amount"] - paysim_work["newbalanceDest"]
)

# One-hot encode transaction type consistently across the full known schema.
# PaySim's type vocabulary is part of the dataset schema; unknown future values
# should be handled explicitly in a production preprocessing pipeline.
paysim_type_dummies = pd.get_dummies(
    paysim_work["type"].fillna("UNKNOWN"),
    prefix="type",
    dtype="int8"
)
paysim_work = pd.concat([paysim_work, paysim_type_dummies], axis=1)

print("Engineered PaySim shape:", paysim_work.shape)
print("Generated type columns:", paysim_type_dummies.columns.tolist())

Engineered PaySim shape: (6362620, 23)
Generated type columns: ['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [13]:
paysim_target = paysim_work["isFraud"].astype("int8").reset_index(drop=True)
paysim_step = paysim_work["step"].reset_index(drop=True)

identifier_cols = [c for c in ["nameOrig", "nameDest"] if c in paysim_work.columns]
target_and_metadata = ["isFraud", "isFlaggedFraud", "step", "type"] + identifier_cols

# Real-time feature set: excludes post-transaction balance fields and derived
# balance changes/errors that depend on new balances.
realtime_exclusions = [
    "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
    "origin_balance_change", "destination_balance_change",
    "origin_balance_error", "destination_balance_error"
]

paysim_X_extended = paysim_work.drop(
    columns=target_and_metadata,
    errors="ignore"
).copy()

paysim_X_realtime = paysim_X_extended.drop(
    columns=realtime_exclusions,
    errors="ignore"
).copy()

# Ensure matrices contain numeric columns only after type one-hot encoding.
for name, frame in [
    ("PaySim extended", paysim_X_extended),
    ("PaySim real-time", paysim_X_realtime)
]:
    non_numeric = frame.select_dtypes(exclude=np.number).columns.tolist()
    if non_numeric:
        raise TypeError(f"{name} contains non-numeric columns: {non_numeric}")

print("PaySim real-time features:", paysim_X_realtime.shape)
print("PaySim extended features:", paysim_X_extended.shape)
print("Target:", paysim_target.shape)
print("Real-time columns:", paysim_X_realtime.columns.tolist())

PaySim real-time features: (6362620, 10)
PaySim extended features: (6362620, 17)
Target: (6362620,)
Real-time columns: ['amount', 'oldbalanceOrg', 'hour_of_day', 'simulation_day', 'log_amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


## 11. PaySim chronological split and save

In [14]:
paysim_train_idx, paysim_val_idx, paysim_test_idx = chronological_split_indices(
    paysim_step,
    TRAIN_FRACTION,
    VALIDATION_FRACTION
)

paysim_splits = {}
for split_name, indices in [
    ("train", paysim_train_idx),
    ("validation", paysim_val_idx),
    ("test", paysim_test_idx)
]:
    X_rt = paysim_X_realtime.iloc[indices].reset_index(drop=True)
    X_ext = paysim_X_extended.iloc[indices].reset_index(drop=True)
    y = paysim_target.iloc[indices].reset_index(drop=True)
    step = paysim_step.iloc[indices].reset_index(drop=True)

    paysim_splits[split_name] = {"realtime": X_rt, "extended": X_ext, "y": y, "step": step}

    print(
        f"{split_name}: {len(indices):,} rows | "
        f"step {step.min()}–{step.max()} | fraud rate {y.mean()*100:.4f}%"
    )

train: 4,453,834 rows | step 1–323 | fraud rate 0.0818%
validation: 954,393 rows | step 323–378 | fraud rate 0.0589%
test: 954,393 rows | step 378–743 | fraud rate 0.4200%


In [15]:
# Save full feature sets for compatibility with the previous workflow.
paysim_X_realtime.to_parquet(PROCESSED_DIR / "paysim_realtime_features.parquet", index=False)
paysim_X_extended.to_parquet(PROCESSED_DIR / "paysim_extended_features.parquet", index=False)
paysim_target.to_frame("isFraud").to_parquet(PROCESSED_DIR / "paysim_target.parquet", index=False)
paysim_step.to_frame("step").to_parquet(PROCESSED_DIR / "paysim_step.parquet", index=False)

for split_name, split in paysim_splits.items():
    split["realtime"].to_parquet(
        PROCESSED_DIR / f"paysim_X_{split_name}_realtime.parquet", index=False
    )
    split["extended"].to_parquet(
        PROCESSED_DIR / f"paysim_X_{split_name}_extended.parquet", index=False
    )
    split["y"].to_frame("isFraud").to_parquet(
        PROCESSED_DIR / f"paysim_y_{split_name}.parquet", index=False
    )
    split["step"].to_frame("step").to_parquet(
        PROCESSED_DIR / f"paysim_step_{split_name}.parquet", index=False
    )

paysim_manifest = {
    "dataset": "PaySim",
    "split_strategy": "chronological row-position split by step",
    "fractions": {"train": TRAIN_FRACTION, "validation": VALIDATION_FRACTION, "test": TEST_FRACTION},
    "rows": {name: len(split["y"]) for name, split in paysim_splits.items()},
    "step_ranges": {
        name: [int(split["step"].min()), int(split["step"].max())]
        for name, split in paysim_splits.items()
    },
    "target": "isFraud",
    "real_time_exclusions": realtime_exclusions,
    "identifier_columns_excluded": identifier_cols
}
with open(REPORT_DIR / "paysim_split_manifest.json", "w") as f:
    json.dump(paysim_manifest, f, indent=2)

print("PaySim feature sets, targets, chronological splits, and manifest saved.")

PaySim feature sets, targets, chronological splits, and manifest saved.


## 12. Split validation and leakage checks

In [16]:
# Verify saved IEEE-CIS split files WITHOUT loading the large feature matrices.
# The Parquet metadata gives row/column counts using very little RAM.
import pyarrow.parquet as pq

for split_name in ["train", "validation", "test"]:
    x_path = PROCESSED_DIR / f"ieee_X_{split_name}.parquet"
    y_path = PROCESSED_DIR / f"ieee_y_{split_name}.parquet"

    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(f"Missing IEEE-CIS files for {split_name}")

    x_metadata = pq.ParquetFile(x_path).metadata
    y_split = pd.read_parquet(y_path, columns=["isFraud"])

    assert x_metadata.num_rows == len(y_split), (
        f"Row mismatch in IEEE-CIS {split_name}: "
        f"X={x_metadata.num_rows}, y={len(y_split)}"
    )
    assert y_split["isFraud"].isin([0, 1]).all()

    print(
        f"IEEE-CIS {split_name}: "
        f"{x_metadata.num_rows:,} rows, "
        f"{len(pq.ParquetFile(x_path).schema.names)} features, "
        f"fraud rate={y_split['isFraud'].mean()*100:.4f}%"
    )
    del y_split

# Validate the already-created PaySim split objects without reloading files.
for split_name, split in paysim_splits.items():
    assert (
        len(split["realtime"])
        == len(split["extended"])
        == len(split["y"])
        == len(split["step"])
    ), f"PaySim row mismatch in {split_name}"

    assert split["y"].isin([0, 1]).all()

assert (
    paysim_splits["train"]["step"].max()
    <= paysim_splits["validation"]["step"].min()
)
assert (
    paysim_splits["validation"]["step"].max()
    <= paysim_splits["test"]["step"].min()
)

assert list(paysim_splits["train"]["realtime"].columns) == list(
    paysim_splits["validation"]["realtime"].columns
)
assert list(paysim_splits["train"]["realtime"].columns) == list(
    paysim_splits["test"]["realtime"].columns
)
assert list(paysim_splits["train"]["extended"].columns) == list(
    paysim_splits["validation"]["extended"].columns
)
assert list(paysim_splits["train"]["extended"].columns) == list(
    paysim_splits["test"]["extended"].columns
)

# Basic feature/target leakage checks.
assert "isFraud" not in paysim_X_realtime.columns
assert "isFraud" not in paysim_X_extended.columns
assert "nameOrig" not in paysim_X_realtime.columns
assert "nameDest" not in paysim_X_realtime.columns

print("All split alignment, chronological-order, and leakage checks passed.")

IEEE-CIS train: 413,378 rows, 432 features, fraud rate=3.5169%
IEEE-CIS validation: 88,581 rows, 432 features, fraud rate=3.4341%
IEEE-CIS test: 88,581 rows, 432 features, fraud rate=3.4804%
All split alignment, chronological-order, and leakage checks passed.


In [17]:
# Summarize all splits in one report.
split_rows = []

for dataset_name, split_dict in [
    ("IEEE-CIS", {
        "train": (ieee_X_train, ieee_y_train),
        "validation": (ieee_X_validation, ieee_y_validation),
        "test": (ieee_X_test, ieee_y_test)
    }),
    ("PaySim", {
        name: (part["realtime"], part["y"])
        for name, part in paysim_splits.items()
    })
]:
    for split_name, (X_part, y_part) in split_dict.items():
        split_rows.append({
            "Dataset": dataset_name,
            "Split": split_name,
            "Rows": len(X_part),
            "Features": X_part.shape[1],
            "Fraud_Count": int(y_part.sum()),
            "Fraud_Rate_Percent": float(y_part.mean() * 100)
        })

split_summary = pd.DataFrame(split_rows)
display(split_summary.round(4))
split_summary.to_csv(REPORT_DIR / "preprocessing_split_summary.csv", index=False)

,Dataset,Split,Rows,Features,Fraud_Count,Fraud_Rate_Percent
0,IEEE-CIS,train,413378,432,14538,3.51690
1,IEEE-CIS,validation,88581,432,3042,3.43410
2,IEEE-CIS,test,88581,432,3083,3.48040
3,PaySim,train,4453834,10,3643,0.08180
4,PaySim,validation,954393,10,562,0.05890
5,PaySim,test,954393,10,4008,0.42000


In [18]:
# Final output inventory and sizes.
for path in sorted(PROCESSED_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(PROCESSED_DIR)} — {path.stat().st_size / (1024**2):.2f} MB")

print("\nNotebook 02 preprocessing and feature engineering completed.")

ieee_X_test.parquet — 12.58 MB
ieee_X_train.parquet — 54.74 MB
ieee_X_validation.parquet — 11.65 MB
ieee_id_test.parquet — 0.51 MB
ieee_id_train.parquet — 1.84 MB
ieee_id_validation.parquet — 0.51 MB
ieee_y_test.parquet — 0.01 MB
ieee_y_train.parquet — 0.03 MB
ieee_y_validation.parquet — 0.01 MB
paysim_X_test_extended.parquet — 47.69 MB
paysim_X_test_realtime.parquet — 17.55 MB
paysim_X_train_extended.parquet — 223.76 MB
paysim_X_train_realtime.parquet — 81.89 MB
paysim_X_validation_extended.parquet — 47.73 MB
paysim_X_validation_realtime.parquet — 17.48 MB
paysim_extended_features.parquet — 317.92 MB
paysim_realtime_features.parquet — 116.63 MB
paysim_step.parquet — 0.03 MB
paysim_step_test.parquet — 0.01 MB
paysim_step_train.parquet — 0.02 MB
paysim_step_validation.parquet — 0.01 MB
paysim_target.parquet — 0.03 MB
paysim_y_test.parquet — 0.01 MB
paysim_y_train.parquet — 0.02 MB
paysim_y_validation.parquet — 0.00 MB
reports/ieee_categorical_cardinality_train.csv — 0.00 MB
reports/ieee